# Eye Tracker Data: First Look — Visualising Raw Streams

This notebook is your starting point for the eye-tracking data workshop series.
Before cleaning or analysing data, you need to know what it looks like.

You will load a real Neon recording, run a few pre-solved helpers, and inspect
the gaze, pupil, blink, and fixation streams visually.


## Learning Goals

By the end, you should be able to:

1. Load a Neon recording and identify its main data streams.
2. Estimate sampling rate, timing stability, and sample completeness for a stream.
3. Visualise gaze as time series and as a spatial trace.
4. Relate Neon blink events to pupil diameter over time.
5. Inspect fixation locations and durations as a first-look map.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from nb_setup import DEFAULT_PARTICIPANT_ID, ensure_workshop_data, setup
PROJECT_ROOT = setup()
ensure_workshop_data(PROJECT_ROOT)

from libs.analysis.recording_helpers import find_neon_recording, load_neon, neon_participant_id

## Pre-solved — Stream Sanity Helper

**Pre-solved for workshop timing.** Run this cell and use it in the later visualisation/analysis sections; no live coding needed here.

Before interpreting gaze patterns, run a lightweight sanity check on the loaded
streams. A useful first pass should answer three questions:

- How many samples are available?
- Is the sampling interval stable enough for the planned analysis?
- What fraction of samples contains finite gaze or pupil values?

The provided `stream_quality_summary` implementation treats a sample as complete
only when the timestamp and all supplied signal values are finite. That is not a
complete eye-tracking quality verdict: Neon blink events live in a separate table
and should be inspected directly.


In [ ]:
def stream_quality_summary(timestamps: np.ndarray, *signals: np.ndarray) -> dict:
    """Summarise sampling and sample completeness for aligned streams.

    Args:
        timestamps: 1-D timestamps in seconds.
        *signals: one or more arrays aligned with timestamps. A sample is complete
                  only if every supplied signal is finite.

    Returns:
        Dict with keys: n_samples, duration_s, median_sampling_rate_hz,
        sampling_jitter_ms, tracking_ratio.
    """
    timestamps = np.asarray(timestamps, dtype=float)
    if timestamps.ndim != 1:
        raise ValueError("timestamps must be a 1-D array")

    finite_time = np.isfinite(timestamps)
    t_finite = timestamps[finite_time]
    if t_finite.size >= 2:
        duration_s = float(t_finite[-1] - t_finite[0])
        dt = np.diff(t_finite)
        dt = dt[dt > 0]
    else:
        duration_s = 0.0
        dt = np.array([], dtype=float)

    if dt.size:
        median_dt = float(np.median(dt))
        median_sampling_rate_hz = float(1.0 / median_dt) if median_dt > 0 else np.nan
        sampling_jitter_ms = float(np.std(dt) * 1000.0)
    else:
        median_sampling_rate_hz = np.nan
        sampling_jitter_ms = np.nan

    if signals:
        valid = finite_time.copy()
        for sig in signals:
            sig = np.asarray(sig, dtype=float)
            if sig.shape != timestamps.shape:
                raise ValueError("all signals must have the same shape as timestamps")
            valid &= np.isfinite(sig)
    else:
        valid = finite_time

    tracking_ratio = float(np.mean(valid)) if timestamps.size else np.nan

    return {
        "n_samples": int(timestamps.size),
        "duration_s": duration_s,
        "median_sampling_rate_hz": median_sampling_rate_hz,
        "sampling_jitter_ms": sampling_jitter_ms,
        "tracking_ratio": tracking_ratio,
    }


In [ ]:
def test_stream_quality_summary(fn):
    t = np.array([0.00, 0.01, 0.02, 0.04])
    x = np.array([1.0, 2.0, np.nan, 4.0])
    y = np.array([1.0, 2.0, 3.0, 4.0])
    out = fn(t, x, y)
    expected_keys = {
        "n_samples", "duration_s", "median_sampling_rate_hz",
        "sampling_jitter_ms", "tracking_ratio",
    }
    assert set(out) == expected_keys, f"Unexpected keys: {set(out)}"
    assert out["n_samples"] == 4, "Expected four samples."
    assert np.isclose(out["duration_s"], 0.04), f"Unexpected duration: {out['duration_s']}"
    assert np.isclose(out["median_sampling_rate_hz"], 100.0),         f"Unexpected median sampling rate: {out['median_sampling_rate_hz']}"
    assert np.isclose(out["sampling_jitter_ms"], np.std([0.01, 0.01, 0.02]) * 1000),         f"Unexpected timing jitter: {out['sampling_jitter_ms']}"
    assert np.isclose(out["tracking_ratio"], 0.75),         f"Expected complete-sample ratio 3/4, got {out['tracking_ratio']}"
    print("Pre-solved stream-sanity helper passed.")


test_stream_quality_summary(stream_quality_summary)


## Pre-solved — Inter-Blink Intervals

**Pre-solved for workshop timing.** Run this cell and use it in the later visualisation/analysis sections; no live coding needed here.

The inter-blink interval (IBI) is the quiet period between consecutive blinks.
It is sensitive to fatigue, task load, and cognitive state.

The provided `inter_blink_intervals` implementation. It receives a blinks DataFrame with
`start_timestamp` and `end_timestamp` columns (in seconds) and returns
an array of IBI values defined as the gap from the **end** of one blink
to the **start** of the next.


In [ ]:
def inter_blink_intervals(blinks: pd.DataFrame) -> np.ndarray:
    """Compute the gap from the end of each blink to the start of the next.

    Args:
        blinks: DataFrame with start_timestamp and end_timestamp columns
                in seconds, sorted by start_timestamp.

    Returns:
        1-D numpy array of IBI values in seconds.
        Length is len(blinks) - 1. Empty array for fewer than 2 blinks.
    """
    if blinks is None or len(blinks) < 2:
        return np.array([], dtype=float)
    blinks = blinks.sort_values("start_timestamp").reset_index(drop=True)
    ends   = blinks["end_timestamp"].to_numpy(dtype=float)
    starts = blinks["start_timestamp"].to_numpy(dtype=float)
    return starts[1:] - ends[:-1]

In [ ]:
def test_inter_blink_intervals(fn):
    blinks_test = pd.DataFrame({
        "start_timestamp": [1.0, 3.0, 6.0],
        "end_timestamp":   [1.2, 3.5, 6.4],
    })
    out = fn(blinks_test)
    assert isinstance(out, np.ndarray), "Return a numpy array."
    assert len(out) == 2, "Three blinks produce two inter-blink intervals."
    assert np.isclose(out[0], 1.8), \
        f"First IBI should be 3.0 - 1.2 = 1.8 s, got {out[0]:.3f}."
    assert np.isclose(out[1], 2.5), \
        f"Second IBI should be 6.0 - 3.5 = 2.5 s, got {out[1]:.3f}."
    single = pd.DataFrame({"start_timestamp": [1.0], "end_timestamp": [1.2]})
    assert fn(single).size == 0, "Single blink should return an empty array."
    print("Pre-solved inter-blink helper passed.")


test_inter_blink_intervals(inter_blink_intervals)


## Load a Real Neon Recording

By default this notebook uses `DEFAULT_PARTICIPANT_ID` from `nb_setup.py` (`p0097`).
Change it to `"p0096"` or `"p0097"` to inspect another subject, or set
`PREFERRED_RECORDING_ID` to a Neon folder UUID to pin a specific session.

In [ ]:
EXPERIMENT = "IMRFSpatialAV"
PARTICIPANT_ID = DEFAULT_PARTICIPANT_ID  # options: "p0096", "p0097", "p0099"
PREFERRED_RECORDING_ID = None # optional Neon folder UUID override

NEON_ROOT = PROJECT_ROOT / "data_output" / EXPERIMENT / "neon"
RECORDING_PATH = find_neon_recording(
    NEON_ROOT,
    participant_id=PARTICIPANT_ID,
    recording_id=PREFERRED_RECORDING_ID,
)
if RECORDING_PATH is None or not RECORDING_PATH.is_dir():
    raise FileNotFoundError(
        f"No Neon recording found under {NEON_ROOT} for participant {PARTICIPANT_ID!r}. "
        "Check PARTICIPANT_ID or set PREFERRED_RECORDING_ID."
    )

SELECTED_PARTICIPANT = neon_participant_id(RECORDING_PATH) or PARTICIPANT_ID
rec = load_neon(str(RECORDING_PATH))
print(f"Participant: {SELECTED_PARTICIPANT}")
print(f"Recording  : {RECORDING_PATH.name}")
print(f"Duration   : {rec.duration_seconds:.1f} s")
print(f"Sample rate: ~{rec.sampling_rate:.0f} Hz")


In [ ]:
assert rec.has_gaze,  "No gaze stream in this recording."
assert rec.has_pupil, "No pupil stream in this recording."

gaze_df  = rec.gaze.copy()
gaze_t   = gaze_df["timestamp"].to_numpy(dtype=float)
gaze_x   = gaze_df["x"].to_numpy(dtype=float)
gaze_y   = gaze_df["y"].to_numpy(dtype=float)

pupil_df = rec.pupil.copy()
pupil_t  = pupil_df["timestamp"].to_numpy(dtype=float)
pupil_l  = pupil_df["diameter_left"].to_numpy(dtype=float)
pupil_r  = pupil_df["diameter_right"].to_numpy(dtype=float)

blinks    = rec.blinks.copy()    if rec.has_blinks          else pd.DataFrame(columns=["start_timestamp", "end_timestamp"])
fixations = rec.fixations.copy() if rec.fixations is not None else pd.DataFrame()
saccades  = rec.saccades.copy()  if rec.saccades  is not None else pd.DataFrame()

print(f"Gaze samples  : {len(gaze_t):,}")
print(f"Pupil samples : {len(pupil_t):,}")
print(f"Blinks        : {len(blinks):,}")
print(f"Fixations     : {len(fixations):,}")
print(f"Saccades      : {len(saccades):,}")
print()
print("Gaze columns  :", list(gaze_df.columns))
print("Pupil columns :", list(pupil_df.columns))
display(gaze_df.head())

## Recording Sanity Check

Before looking at visual patterns, run a lightweight sanity check. The goal is
not to certify the recording as "good" in one number, but to catch obvious
problems before you start interpreting plots.

This section checks stream duration, sample counts, timestamp order, sampling
rate, sample completeness, and blink event rate. The blink values come from the
Neon `blinks` event stream, not from missing samples in the gaze or pupil arrays.


In [ ]:
sanity = {
    "recording_duration_s": round(rec.duration_seconds, 2),
    "gaze_samples": len(gaze_t),
    "pupil_samples": len(pupil_t),
    "blink_events": len(blinks),
    "blink_rate_per_min": np.nan,
    "gaze_time_monotonic": bool(np.all(np.diff(gaze_t) > 0)) if len(gaze_t) > 1 else False,
    "pupil_time_monotonic": bool(np.all(np.diff(pupil_t) > 0)) if len(pupil_t) > 1 else False,
}
if rec.duration_seconds > 0:
    sanity["blink_rate_per_min"] = round(len(blinks) / (rec.duration_seconds / 60.0), 1)

display(pd.DataFrame.from_dict(sanity, orient="index", columns=["value"]))

quality_rows = []
for name, timestamps, signals in [
    ("gaze xy", gaze_t, (gaze_x, gaze_y)),
    ("pupil both eyes", pupil_t, (pupil_l, pupil_r)),
]:
    q = stream_quality_summary(timestamps, *signals)
    quality_rows.append({
        "stream": name,
        "samples": q["n_samples"],
        "duration_s": round(q["duration_s"], 2),
        "median_hz": round(q["median_sampling_rate_hz"], 2),
        "jitter_ms": round(q["sampling_jitter_ms"], 3),
        "complete_samples_%": round(q["tracking_ratio"] * 100.0, 1),
        "incomplete_samples_%": round((1.0 - q["tracking_ratio"]) * 100.0, 1),
    })

quality_df = pd.DataFrame(quality_rows).set_index("stream")
display(quality_df)


## Gaze Position over Time

The two panels show gaze x and y as time series. Look for:

- **Sudden vertical jumps** — saccades between fixations.
- **Flat regions near a fixed value** — gaze held still during a fixation.
- **Possible artifacts** — impossible jumps, flatlines, or out-of-range samples.

Use the blink-window plot below to compare the gaze trace with detected blink
events.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(gaze_t, gaze_x, lw=0.6, alpha=0.75, color="C0")
axes[0].set_ylabel("gaze x")
axes[0].set_title("Gaze position over time")

axes[1].plot(gaze_t, gaze_y, lw=0.6, alpha=0.75, color="C1")
axes[1].set_ylabel("gaze y")
axes[1].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

## Gaze Trace (Spatial View)

Plotting gaze x against gaze y shows a spatial picture of where the participant
looked. Colour encodes normalised time so you can follow the sequence of
fixations across the scene.

The y-axis is inverted because screen coordinates start from the top-left corner.

In [ ]:
finite  = np.isfinite(gaze_x) & np.isfinite(gaze_y)
gx_f, gy_f, gt_f = gaze_x[finite], gaze_y[finite], gaze_t[finite]
t_norm  = (gt_f - gt_f.min()) / max(gt_f.max() - gt_f.min(), 1e-9)

fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(gx_f, gy_f, c=t_norm, cmap="plasma", s=1.0, alpha=0.35, linewidths=0)
plt.colorbar(sc, ax=ax, label="Normalised time  (0 = start,  1 = end)")
ax.set_xlabel("gaze x")
ax.set_ylabel("gaze y")
ax.set_title("Gaze trace  (colour = time)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Exercise 1 — Flag Blink-Contaminated Samples

Each Neon blink event has a `start_timestamp` and an `end_timestamp`.
To exclude or annotate blink-contaminated gaze or pupil samples, you need
to know **which samples overlap with any blink window**.

Complete `flag_blink_samples`. It should return a boolean array — `True`
for every sample whose timestamp falls inside at least one blink interval,
`False` otherwise.

The scaffold below shows the pattern: iterate over blink rows, build a
per-blink boolean mask with `>=` and `<=`, and combine them with `|=`.

In [ ]:
def flag_blink_samples(
    timestamps: np.ndarray,
    blinks: pd.DataFrame,
) -> np.ndarray:
    """Return a boolean mask: True for samples that fall inside a blink window.

    Args:
        timestamps: 1-D array of sample timestamps in seconds.
        blinks: DataFrame with start_timestamp and end_timestamp columns.

    Returns:
        Boolean array of the same length as timestamps.
        All False when blinks is empty or None.
    """
    # Starter scaffold — fill in the ... blanks, then uncomment:
    # mask = np.zeros(len(...), dtype=bool)
    # if blinks is ...:
    #     return mask
    # for _, blink in blinks.iterrows():
    #     s = float(blink["..."])
    #     e = float(blink["..."])
    #     mask |= (timestamps >= s) & (timestamps <= e)
    # return mask
    raise NotImplementedError("Complete flag_blink_samples")

In [ ]:
def test_flag_blink_samples(fn):
    ts = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
    blinks_test = pd.DataFrame({
        "start_timestamp": [0.4, 2.2],
        "end_timestamp":   [0.8, 2.6],
    })
    out = fn(ts, blinks_test)
    expected = np.array([False, True, False, False, False, True, False])
    assert isinstance(out, np.ndarray), "Return a numpy array."
    assert out.dtype == bool, "The mask must be boolean."
    assert out.shape == ts.shape, "Output shape must match timestamps shape."
    assert np.array_equal(out, expected), f"Expected {expected}, got {out}."
    # Boundary: sample exactly on start/end is inside
    boundary = fn(np.array([0.4, 0.8]), blinks_test)
    assert boundary.all(), "Boundary timestamps must be flagged."
    # Empty blinks → all False
    empty = pd.DataFrame(columns=["start_timestamp", "end_timestamp"])
    assert not fn(ts, empty).any(), "No blinks → all False."
    print("Exercise 1 passed.")


test_flag_blink_samples(flag_blink_samples)

In [ ]:
# How many pupil samples are blink-contaminated?
blink_mask_pupil = flag_blink_samples(pupil_t, blinks)
n_blink = int(blink_mask_pupil.sum())
pct     = 100.0 * blink_mask_pupil.mean()
print(f"Pupil samples inside blink windows: {n_blink:,} / {len(pupil_t):,}  ({pct:.1f} %)")

## Pupil Diameter with Blink Windows

The blink-detection algorithm records start and end times for each blink;
these are shaded gold below. Inspect whether pupil diameter changes inside each
blink window, and whether blink periods create artifacts that would matter for
your later analysis.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))

ax.plot(pupil_t, pupil_l, lw=0.8, label="left",  alpha=0.9, color="C0")
ax.plot(pupil_t, pupil_r, lw=0.8, label="right", alpha=0.7, color="C1")

if not blinks.empty:
    for _, blink in blinks.iterrows():
        s = float(blink.get("start_timestamp", np.nan))
        e = float(blink.get("end_timestamp",   np.nan))
        if not np.isfinite(s):
            continue
        if not np.isfinite(e) or e <= s:
            e = s + 0.15
        ax.axvspan(s, e, color="gold", alpha=0.18, linewidth=0)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Pupil diameter")
ax.set_title("Pupil diameter (left & right) with blink windows shaded")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Blink Characterisation

The two histograms below use the pre-solved inter-blink helper:

- **Blink duration**: how long each blink lasted (typically 100-400 ms).
- **Inter-blink interval**: the quiet period between consecutive blinks
  (varies with task and fatigue).


In [ ]:
if blinks.empty:
    print("No blink events in this recording.")
else:
    dur_ms  = (blinks["end_timestamp"] - blinks["start_timestamp"]).clip(lower=0.0) * 1000.0
    ibis_ms = inter_blink_intervals(blinks) * 1000.0
    ibis_ms = ibis_ms[ibis_ms > 0]  # drop negatives from any out-of-order rows

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.hist(dur_ms, bins=25, edgecolor="white", color="C4")
    ax1.set_xlabel("Blink duration (ms)")
    ax1.set_ylabel("Count")
    ax1.set_title(
        f"Blink durations  (n = {len(dur_ms)})\n"
        f"median = {dur_ms.median():.0f} ms"
    )

    ax2.hist(ibis_ms, bins=25, edgecolor="white", color="C2")
    ax2.set_xlabel("Inter-blink interval (ms)")
    ax2.set_ylabel("Count")
    ax2.set_title(
        f"Inter-blink intervals  (n = {len(ibis_ms)})\n"
        f"median = {np.median(ibis_ms):.0f} ms"
    )

    plt.tight_layout()
    plt.show()


## Fixation Map

Each circle marks the location of one fixation. Its area is proportional to
duration — larger circles indicate longer dwells. The plot shows which regions
of the scene attracted the most sustained attention.

In [ ]:
if fixations.empty:
    print("No fixation events in this recording.")
else:
    print("Fixation columns:", list(fixations.columns))

    # The Neon SDK labels the centroid position differently across versions.
    x_candidates = ["x", "mean_gaze_x", "start_gaze_x"]
    y_candidates = ["y", "mean_gaze_y", "start_gaze_y"]
    fx_col = next((c for c in x_candidates if c in fixations.columns), None)
    fy_col = next((c for c in y_candidates if c in fixations.columns), None)

    if fx_col is None or fy_col is None:
        print("Cannot find centroid columns — available:", list(fixations.columns))
    else:
        dur_ms = (
            fixations["end_timestamp"] - fixations["start_timestamp"]
        ).clip(lower=0.0) * 1000.0
        fx = fixations[fx_col].to_numpy(dtype=float)
        fy = fixations[fy_col].to_numpy(dtype=float)

        fig, ax = plt.subplots(figsize=(7, 5.5))
        ax.scatter(fx, fy, s=dur_ms / 5.0, alpha=0.55,
                   edgecolors="white", linewidths=0.4)
        ax.set_xlabel(fx_col)
        ax.set_ylabel(fy_col)
        ax.set_title(f"Fixations  (n = {len(fixations)}) — area ∝ duration")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()


## Short Reflection

Answer these in a markdown cell below, or discuss as a group:

1. Was the median sampling rate close to what the recording metadata reported?
   Was the timing jitter small enough for the analyses you plan to run?
2. Were the gaze and pupil streams complete and timestamp-monotonic? Would you
   trust them for visualisation, preprocessing, or trial-level analysis?
3. What was the approximate blink rate (blinks per minute)? How might the task
   context explain a value below the typical resting blink rate of 15-20/min?
4. Which region of the scene received the most fixations?
   Does that match what you would expect from the task design?
5. What would you check before using this recording in an analysis pipeline?
